# RuralPulse v2: walkthrough
This notebook calls the same engine as the website. It runs **without an API key**: rules-only mode is the default. Add `GOOGLE_API_KEY` to `../.env` to let Gemini assist understanding.

See `../docs/evidence_review.md` for why the design is safety-first.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('RP_AUDIT_LOG', '0')
from backend.app.pipeline import interpret, assess
from backend.app.schemas import InterpretRequest, AssessRequest
from backend.app import llm
print('Gemini enabled:', llm.available())

## 1. Emergency: rules fire before any model call

In [ ]:
r = interpret(InterpretRequest(message='அப்பாவுக்கு நெஞ்சு வலி, வியர்க்கிறது', at_time='2026-10-07T11:00'))
print(r['stage'], '|', r['urgency']['title'])
print([x['id'] for x in r['decision']['rules_fired']], '| LLM:', r['interpretation']['llm']['status'])
print('Go to:', r['navigation']['primary']['name'])

## 2. Routine case: read-back, then confirm

In [ ]:
req = dict(message='3 din se bukhar aur khansi hai', at_time='2026-10-07T11:00')
r = interpret(InterpretRequest(**req))
print(r['readback'])
for q in r['questions']: print('-', q['id'], ':', q['text'])

In [ ]:
a = assess(AssessRequest(**req, interpretation_id=r['interpretation_id'], answers={'age': 34, 'redflag_screen': 'no'}, confirmed=True))
print(a['urgency']['title'], [x['id'] for x in a['decision']['rules_fired']])
print()
print(a['summary']['doctor_text'])
print()
print('SMS:', a['summary']['sms'])

## 3. Abstention: an untested language and a dose question

In [ ]:
for m in ['Mor buka bahut dukhena, tini dina heli', 'How many tablets of paracetamol should I take?']:
    x = interpret(InterpretRequest(message=m))
    if x['stage'] == 'confirm':
        x = assess(AssessRequest(message=m, interpretation_id=x['interpretation_id'], answers={'redflag_screen':'no'}, confirmed=True))
    print(m, '->', x['urgency']['level'], x['decision']['abstain_reasons'])

## 4. Verified navigation at night

In [ ]:
m = 'Fever for 5 days, age 40'
x = interpret(InterpretRequest(message=m, at_time='2026-10-07T22:00'))
x = assess(AssessRequest(message=m, interpretation_id=x['interpretation_id'], answers={'redflag_screen':'no'}, confirmed=True, at_time='2026-10-07T22:00'))
n = x['navigation']
print('Primary:', n['primary']['name'], '|', n['primary']['open']['text'], '|', n['primary']['freshness'])
print('Warnings:', n['warnings'])
print('Excluded:', [(e['id'], e['reason']) for e in n['excluded']][:5])

## 5. Safety suite

In [ ]:
from backend.app.evaluation import run_suite
res = run_suite()
print(json.dumps(res['metrics'], indent=1))
print(res['under_triaged'])